# Session 2 — Demo 2 — Joins: the warehouse gets a geography

**What this demo covers.** Events alone don't know *where* a cell is or *what* an item weighs; that lives in other tables. Tonight we learn the verb that combines tables: the **join** (`merge` in pandas). We start on two toy tables of a few rows, covering all four join types and what empty or duplicated keys do. Then we join the real warehouse and meet the three ways a join can go wrong while looking perfectly fine:

1. lost rows, 2. multiplied rows, 3. wrong key

The first two change the row count, so one habit catches both: **count your rows before and after.** The third one keeps the count intact, and we'll see what to do about that.

At the end there is an **⚡ Extras** section with pro shortcuts. Optional.

**Data.** Same ALPHA week + the reference tables (`topology`, `ovh`, `item_tags`) — [data dictionary](../../data/README.md).

**How to run.** Locally: top to bottom. Colab: uncomment the lines in the setup cell.

## 0. Setup

In [1]:
from pathlib import Path

import pandas as pd

# where the data lives, relative to this notebook
DATA = Path("../../data/alpha")

# --- Google Colab? Uncomment these two lines to fetch the course repo: ---
# !git clone https://github.com/seemsGoodNow/hse-data-science-course.git course
# DATA = Path("course/data/alpha")

# missing a library? uncomment:
# %pip install pandas pyarrow

In [2]:
events = pd.read_parquet(DATA / "picking_events.parquet")
topology = pd.read_parquet(DATA / "topology.parquet")
ovh = pd.read_parquet(DATA / "ovh.parquet")
item_tags = pd.read_parquet(DATA / "item_tags.parquet")

print(
    f"events {len(events):,}, topology {len(topology):,}, "
    f"ovh {len(ovh):,}, item_tags {len(item_tags):,}"
)

# a cell shows only its LAST value on its own; display() shows a table at any point
display(events.head(3))
display(topology[["cell_id", "rack_id", "sector_id", "zone_id"]].head(3))
display(ovh.head(3))
display(item_tags.head(3))

events 311,543, topology 28,805, ovh 60,000, item_tags 60,000


,event_time,event_type,user_id,task_id,cell_id,boxing_id,item_id,items_in_cell_quantity,reason
0,2026-03-02 11:51:59.142,StartTask,4500859,800000001,NaN,NaN,NaN,NaN,NaN
1,2026-03-02 11:53:33.779,ScanCell,4500859,800000001,54400900.0,NaN,NaN,135.0,NaN
2,2026-03-02 11:53:47.415,PickWrongItem,4500859,800000001,54400900.0,10000001.0,3.014677e+09,135.0,WRONG_ITEM


,cell_id,rack_id,sector_id,zone_id
0,65696855,98326021,5761.0,104959650
1,42773994,98326021,5761.0,104959650
2,160271283,98326021,5761.0,104959650


,item_id,weight,volumeliter
0,318455877,2.571,2.380
1,913398911,0.055,0.271
2,555848141,0.081,0.376


,item_id,tags
0,318455877,[]
1,913398911,[SmallParts]
2,555848141,[]


Before joining anything, say out loud what **one row** of each table is. Every question tonight is really a question about that sentence:

| table | one row is | it answers |
|---|---|---|
| `events` | one scan of a handheld terminal | what happened, when, by whom |
| `topology` | one storage cell | where is this cell? |
| `ovh` | one item in the catalogue | what does this item weigh? |
| `item_tags` | one item in the catalogue | how must this item be handled? |

`topology` has twelve columns; we show four. The rest (coordinates, rack size, which way the rack opens) are in the [data dictionary](../../data/README.md) when you need them.

## 1. What a join is — on six rows

A join glues two tables together by a **key**: a column that means the same thing in both tables.

**You already do this in Excel. It is VLOOKUP.** That is the whole idea, and it is worth carrying the Excel picture with you, because two differences are exactly where tonight's mistakes live:

- VLOOKUP returns the **first** match it finds. A join returns **every** match.
- VLOOKUP only ever **adds** columns. A join can also **delete rows**.

Meet the smallest warehouse again, the same one from Demo 1: six picks by Anna, Boris and Clara, and a reference of four cells. Read a row as a sentence — *"Anna picked a mug from cell A, and it took her 5 seconds."*

The imperfections are deliberate, and they are the whole point: one pick came from cell `D`, which is missing from the reference; one pick has an **empty** cell id, because the scanner never read the shelf label; and the reference knows a cell `E` that nobody picked from all week.

> 🖼 The same six rows and all four results are on **card 3** in `concept-cards.html` (this folder) — the picture to keep open while we run them.

And notice the move itself: we are about to learn four join types on **six rows we typed by hand**, not on 179 thousand. That's the toy-table habit from Demo 1 — when a function is new, shrink the data until you can check the answer by eye.

In [3]:
# the same little warehouse as Demo 1, now with the cell each pick came from
toy_picks = pd.DataFrame({
    "worker":  ["Anna", "Anna", "Boris", "Clara", "Boris", "Clara"],
    "item":    ["mug", "kettle", "mug", "lamp", "mug", "cable"],
    "cell_id": ["A", "A", "B", "C", "D", None],
    "seconds": [5, 7, 6, 2, 9, 3],
})
toy_cells = pd.DataFrame({
    "cell_id": ["A", "B", "C", "E"],
    "zone":    ["Z1", "Z1", "Z2", "Z3"],
})
toy_picks

,worker,item,cell_id,seconds
0,Anna,mug,A,5
1,Anna,kettle,A,7
2,Boris,mug,B,6
3,Clara,lamp,C,2
4,Boris,mug,D,9
5,Clara,cable,NaN,3


In [4]:
toy_cells

,cell_id,zone
0,A,Z1
1,B,Z1
2,C,Z2
3,E,Z3


The pandas verb is `merge`: `left_table.merge(right_table, on="key", how="inner")`. The `how` argument is the **join type**. It decides which rows survive when a key has no partner on the other side. There are four types: same two tables, four different answers. Watch the row counts.

**`inner` — keep only rows whose key matched on both sides.** This is the **default**:

In [5]:
toy_picks.merge(toy_cells, on="cell_id", how="inner")

# or you can use pd.merge(toy_picks, toy_cells, on='cell_id')

,worker,item,cell_id,seconds,zone
0,Anna,mug,A,5,Z1
1,Anna,kettle,A,7,Z1
2,Boris,mug,B,6,Z1
3,Clara,lamp,C,2,Z2


Six picks in → **4 rows out**. Boris's pick from cell `D` is gone (no such cell in the reference), Clara's pick with the empty cell id is gone (**an empty key never matches anything**), and cell `E` never appears (no pick touched it). Nothing on screen tells you two picks are missing.

**`left` — keep ALL rows of the left table**; where the key found no partner, the new columns stay empty:

In [6]:
toy_picks.merge(toy_cells, on="cell_id", how="left")

,worker,item,cell_id,seconds,zone
0,Anna,mug,A,5,Z1
1,Anna,kettle,A,7,Z1
2,Boris,mug,B,6,Z1
3,Clara,lamp,C,2,Z2
4,Boris,mug,D,9,NaN
5,Clara,cable,NaN,3,NaN


**6 rows in, 6 rows out** — nothing lost; Boris's `D` pick and Clara's empty-key pick got an empty `zone` instead of vanishing. This is VLOOKUP's behaviour, and it is why `left` is the analyst's default: honest gaps instead of deletions you never see.

**`right`** is the mirror — keep all rows of the *right* table (now the never-picked cell `E` shows up, with no pick attached). **`outer`** keeps everything from both sides:

In [7]:
right = toy_picks.merge(toy_cells, on="cell_id", how="right")
outer = toy_picks.merge(toy_cells, on="cell_id", how="outer")

print("right:", len(right), "rows")
print("outer:", len(outer), "rows")
outer

right: 5 rows
outer: 7 rows


,worker,item,cell_id,seconds,zone
0,Anna,mug,A,5.0,Z1
1,Anna,kettle,A,7.0,Z1
2,Boris,mug,B,6.0,Z1
3,Clara,lamp,C,2.0,Z2
4,Boris,mug,D,9.0,NaN
5,NaN,NaN,E,NaN,Z3
6,Clara,cable,NaN,3.0,NaN


| `how=` | keeps | on our toy (6 picks × 4 cells) |
|---|---|---|
| `inner` | only matched keys — **the default!** | 4 rows — two picks gone, nothing said |
| `left` | all left rows; gaps where unmatched | 6 rows — gaps visible |
| `right` | all right rows | 5 rows — unpicked `E` appears |
| `outer` | everything from both sides | 7 rows — every gap visible |

In practice you'll rarely write a `right` join: put the table you care about on the left and use `left` instead.

Two rules worth memorizing: **the default is `inner`**, and **an empty key never matches** — not even another empty key. In Excel terms: only `left` behaves like VLOOKUP, and the default does not.

*(One detail in the `outer` result: `seconds` now prints as `5.0` instead of `5`. Cell `E` has no pick, so its `seconds` is empty — and a column with an empty value becomes a float. Same thing you saw in the id columns last week.)*

**The row-count habit, automated.** Python has a one-word tripwire: `assert condition` does nothing while the condition is true and **stops the notebook** the moment it's false. One line after every join:

In [8]:
picks_w_topology_toy = toy_picks.merge(toy_cells, on="cell_id", how="left")

# a left join must not change my row count — stop the notebook if it did
assert len(picks_w_topology_toy) == len(toy_picks)
print("row count preserved:", len(toy_picks), "→", len(picks_w_topology_toy))

row count preserved: 6 → 6


One more toy before the real thing. What if the *reference* side has the same key twice? That is not exotic: a cell holds more than one kind of item, so a table of "what is stored where" has cell `A` twice.

In [9]:
toy_cell_items = pd.DataFrame({
    "cell_id":     ["A", "A", "B"],
    "stored_item": ["mug", "kettle", "mug"],
})
picks_w_items = toy_picks.merge(toy_cell_items, on="cell_id", how="left")
print(f"rows before: {len(toy_picks)}   rows after: {len(picks_w_items)}")
picks_w_items

rows before: 6   rows after: 8


,worker,item,cell_id,seconds,stored_item
0,Anna,mug,A,5,mug
1,Anna,mug,A,5,kettle
2,Anna,kettle,A,7,mug
3,Anna,kettle,A,7,kettle
4,Boris,mug,B,6,mug
5,Clara,lamp,C,2,NaN
6,Boris,mug,D,9,NaN
7,Clara,cable,NaN,3,NaN


Six picks in, **8 rows out**: each pick from cell `A` matched *both* of its stored items and now exists twice. A join doesn't pick one match — it produces **every combination**. This is the second Excel difference, live: VLOOKUP would have taken the first row and moved on. Count picks per cell now, and `A` is counted twice. Remember this table; it comes back on 179 thousand rows. (**Card 5** in `concept-cards.html` is this exact picture.)

## 2. Meet the real cast, check the keys

`topology` — one row per cell, 28,805 of them; `ovh` — one row per item, 60,000 of them. Keys: `cell_id`, `item_id`. After the toy you know what a duplicated key does, so before any join, two seconds of insurance: is the key **unique** on the reference side? Every pandas column has an `.is_unique` attribute that answers exactly that, with no arguments and no ceremony:

In [10]:
print("cell_id unique in topology:", topology["cell_id"].is_unique)
print("item_id unique in ovh:     ", ovh["item_id"].is_unique)
print("item_id unique in tags:    ", item_tags["item_id"].is_unique)

cell_id unique in topology: True
item_id unique in ovh:      True
item_id unique in tags:     True


## 3. First real join — and the row-count discipline

Attach the geography to every pick: `how="left"` keeps **all** my picks and adds location columns where the cell matched. Row counts said out loud, plus the tripwire. *(Small print: `.copy()` after the filter makes `picks` an independent table — we'll add a column to it later, and pandas complains if you write into a filtered slice.)*

In [11]:
picks = events[events["event_type"].isin(["PickCorrectItem", "PickWrongItem"])].copy()

picks_w_topology = picks.merge(topology, on="cell_id", how="left")
print(f"rows before: {len(picks):,}   rows after: {len(picks_w_topology):,}")
assert len(picks_w_topology) == len(picks)

# the join added 11 columns; show the key plus the two we came for
(
    picks_w_topology
    [["event_time", "user_id", "cell_id", "sector_id", "zone_id"]]
    .head()
)

rows before: 179,221   rows after: 179,221


,event_time,user_id,cell_id,sector_id,zone_id
0,2026-03-02 11:53:47.415,4500859,54400900.0,6794.0,121029085
1,2026-03-02 11:53:56.955,4500859,54400900.0,6794.0,121029085
2,2026-03-02 11:54:44.531,4500859,43583220.0,6794.0,121029085
3,2026-03-02 11:55:25.176,4500859,100180332.0,6794.0,121029085
4,2026-03-02 11:55:58.234,4500859,119816855.0,6794.0,121029085


**Same number of rows**, and `assert` passed without a word. Say those two numbers out loud after *every* join, and let `assert` say them for you when you stop trusting yourself to remember.

The join actually added all eleven topology columns, so the full table is twenty columns wide and unreadable on a screen. Selecting the few we came for is not cosmetic: it is how you check that the join did what you meant. One pick, and now it knows its sector and its zone.

Payoff in one line. One new word in it, `.rename`, which gives a column a name — `.size()` returns a column of counts with no name at all, and a nameless column of numbers next to a column of ids is a puzzle nobody needs:

In [12]:
# .size() returns an unnamed column of counts; .rename says what they count
(
    picks_w_topology
    .groupby("zone_id")
    .size()
    .rename("picks")
)

zone_id
104959650    92332
121029085    86889
Name: picks, dtype: int64

In [13]:
# by sector, sorted: where does the work actually happen?
(
    picks_w_topology
    .groupby("sector_id")
    .size()
    .rename("picks")
    .sort_values(ascending=False)
)

sector_id
5400.0    44465
8089.0    30784
5130.0    28168
6794.0    23661
5659.0    19796
5402.0    18763
5761.0    13584
Name: picks, dtype: int64

One zone is ~6% busier — a fact nobody could see in the events table alone. (Whether that's demand or layout: hold the thought, it returns next week with a second warehouse.)

## 4. Silent failure #1 — lost rows

Now attach item weights, but to **all** events, and let's genuinely forget the `how=` argument, the way you would at 19:00 on a Friday. Before running, remember the detective's map: `item_id` is empty on every scan, start and finish row. Predict the row count.

In [14]:
# no how= at all, so pandas uses its default
with_weights = events.merge(ovh, on="item_id")

print(
    f"rows before: {len(events):,}   "
    f"rows after: {len(with_weights):,}   "
    f"vanished: {len(events) - len(with_weights):,}"
)

rows before: 311,543   rows after: 172,189   vanished: 139,354


**Almost half the table vanished**: every row with an empty `item_id` (all scans, box drops, starts, finishes, and the unresolved wrong picks). The cell runs green and hands you a smaller table. It's the toy rule at full scale: `inner` keeps only matched keys, and an empty key matches nothing. This is the failure Excel never had: VLOOKUP could give you `#N/A`, but it could not remove 139 thousand rows and say nothing.

Sometimes that's what you want. It should never be what you get *by accident*.

The honest version: `how="left"` plus one more insurance flag. `indicator=True` adds a service column `_merge` that says, for every row, whether its key matched (`both`) or found no partner (`left_only`) — you *see* what didn't match instead of losing it:

In [15]:
with_weights = events.merge(
    ovh,
    on="item_id",
    how="left",
    indicator=True,   # adds a _merge column saying where each row came from
)
with_weights["_merge"].value_counts()

_merge
both          172189
left_only     139354
right_only         0
Name: count, dtype: int64

## 5. Silent failure #2 — multiplied rows

The third reference, `item_tags`, stores a **list** of tags per item — many items have none, some have several:

In [16]:
item_tags.head(3)

,item_id,tags
0,318455877,[]
1,913398911,[SmallParts]
2,555848141,[]


In [17]:
item_tags["tags"].head(3)

0              []
1    [SmallParts]
2              []
Name: tags, dtype: object

Look at the values: each one is a **list**, not a single value. Pandas can store anything in a column, and when the values are not a standard type (int, str, ...) the column's dtype shows up as `object`.

A list inside a cell is awkward. You can't group by it, you can't compare it with `==`, and counting values counts whole lists rather than tags. So the first move is always to flatten it.

The standard move is `explode`: **one row per element of the list** — an item with two tags becomes two rows, an item with none leaves one empty row behind.

It's a new function, so: the habit again. Two rows we can check by eye, *then* the 60,000-row table. And this time the toy is made of the same stuff as the real table, so nothing has to be translated in your head:

In [18]:
toy_item_tags = pd.DataFrame({
    "item": ["mug", "lamp"],
    "tags": [["Fragile", "Liquid"], ["Oversized"]],
})
toy_item_tags

,item,tags
0,mug,"[Fragile, Liquid]"
1,lamp,[Oversized]


In [19]:
toy_item_tags.explode("tags")

,item,tags
0,mug,Fragile
0,mug,Liquid
1,lamp,Oversized


Two rows became three: the mug's two tags each got their own row. That is the growth we came for.

Back to the warehouse, and in two steps, because two different things happen and only one of them is `explode`:

In [20]:
tags_exploded = item_tags.explode("tags")

print(
    f"rows in item_tags: {len(item_tags):,}   "
    f"after explode: {len(tags_exploded):,}"
)
tags_exploded.head()

rows in item_tags: 60,000   after explode: 61,566


,item_id,tags
0,318455877,NaN
1,913398911,SmallParts
2,555848141,NaN
3,2372181000,NaN
4,518886137,NaN


The table **grew** by 1,566 rows, exactly as the toy promised: every item with two tags now takes two rows.

But most items carry no tags at all, and `explode` leaves each of those as one row with an empty `tags`. Those are not (item, tag) pairs, so we drop them. `dropna(subset=["tags"])` removes rows whose `tags` is empty, and ignores every other column while deciding:

In [21]:
tags_long = tags_exploded.dropna(subset=["tags"])

print(
    f"after explode: {len(tags_exploded):,}   "
    f"after dropping items with no tags: {len(tags_long):,}"
)
tags_long.head()

after explode: 61,566   after dropping items with no tags: 16,059


,item_id,tags
1,913398911,SmallParts
12,1800073444,Fragile
14,2933950321,MultiPack
19,1048282345,HighValue
26,1614107261,Liquid


Note what just happened: `item_id` is **no longer unique** in `tags_long`. You've seen this movie on the toy tables — join it to picks and the row count *must* grow:

In [22]:
picks_tagged = picks.merge(tags_long, on="item_id", how="left")

print(
    f"rows before: {len(picks):,}   "
    f"rows after: {len(picks_tagged):,}   "
    f"appeared from nowhere: {len(picks_tagged) - len(picks):,}"
)

rows before: 179,221   rows after: 183,198   appeared from nowhere: 3,977


The table **grew**. Every pick of a multi-tag item is now counted several times, so every sum and average downstream is wrong. And again, nothing complained.

**An `assert` on that join would have stopped the notebook, and this is what the tripwire looks like when it fires:**

```python
picks_tagged = picks.merge(tags_long, on="item_id", how="left")
assert len(picks_tagged) == len(picks)
```
```
AssertionError                       Traceback (most recent call last)
Cell In[1], line 2
      1 picks_tagged = picks.merge(tags_long, on="item_id", how="left")
----> 2 assert len(picks_tagged) == len(picks)

AssertionError:
```

No message, no advice, just a stop. That is the entire feature. Everything below this line in your notebook stays unrun until you look at the number. Don't run it here — we need the cells below.

The fix here is to not join at all. We care about *one* tag: `SmallParts` (small, light items; many of them share one cell and their packaging barely differs). Take the item ids that carry the tag, and derive a mask on picks with `.isin()` from Demo 1:

In [23]:
small_parts_ids = tags_long[tags_long["tags"] == "SmallParts"]["item_id"]
picks["has_small_parts"] = picks["item_id"].isin(small_parts_ids)
print(f"rows: {len(picks):,} (unchanged — no join happened)")
picks["has_small_parts"].value_counts()

rows: 179,221 (unchanged — no join happened)


has_small_parts
False    167233
True      11988
Name: count, dtype: int64

One row per pick, as before, plus a clean True/False column. When the other table answers a yes/no question, a mask via `.isin()` beats a join: nothing can multiply, nothing can vanish. (Need *several* tags as columns at once? That's an Extras move — see the end.)

## 6. Silent failure #3 — the wrong key

The first two failures changed the row count, so counting rows caught them. The third one doesn't, and that makes it the nastiest.

Our data dictionary says one thing twice, in bold: **ids are local to a warehouse.** ALPHA's `cell_id` 54400900 and BRAVO's `cell_id` 54400900 are two different shelves in two different cities that happen to have been handed the same number. So this join is meaningless. Watch what pandas does about that:

In [24]:
# the OTHER warehouse's cell reference — the same column name, a different meaning
bravo_topology = pd.read_parquet(Path("../../data/bravo") / "topology.parquet")

wrong_key = picks.merge(bravo_topology, on="cell_id", how="inner")
print(f"rows: {len(wrong_key):,}   no error, no warning")
wrong_key["zone_id"].value_counts()

rows: 125   no error, no warning


zone_id
60259905    84
40082688    41
Name: count, dtype: int64

125 rows came back, and the zone ids in them do not exist in ALPHA at all. The key matched by **coincidence**: a handful of numbers happen to be used at both sites. Nothing in pandas can tell the difference between a key that matches and a key that *means the same thing*, because only you know what the column means.

That is the third failure, and it is the reason the two rules on card 4 are not enough on their own. Row counts catch lost and multiplied rows. For the wrong key there is only one defence: **before you join, say out loud what the key means on each side.** If the sentence sounds odd, stop.

## 7. A catalogue is not the traffic

Back to ALPHA. Flip the direction: start from the catalogue and ask which items actually moved this week. `drop_duplicates()` does what it says — collapses repeated rows, leaving each picked item once. Here we *do* want a join rather than a mask, because we want to count both sides at once:

In [25]:
moved = picks[["item_id"]].drop_duplicates()
catalogue = ovh.merge(
    moved,
    on="item_id",
    how="left",
    indicator=True,
)
catalogue["_merge"].value_counts()

_merge
both          31417
left_only     28583
right_only        0
Name: count, dtype: int64

`both` = items picked at least once this week; `left_only` = items that sat untouched. Of 60,000 items, only about half (~31 thousand) moved. Any per-item statistic computed from one week describes *the traffic*, not *the catalogue*. That's the classic trap behind a question like "what is the average weight of our goods?".

## Try it yourself

1. **Picks per shelf level** (`cell_level_in_rack` arrives with the topology join): which level works hardest? Levels 1 and 5–6 are floor and ladder territory — worth knowing before you ever compare picker speeds.
2. What's the **heaviest item picked** this week? *(join picks to `ovh`, sort, head — and think: which `how=` do you want?)*
3. What **share of picks** lands on `Oversized` items? *(same move as `has_small_parts`: filter `tags_long`, `.isin()`, mean of the mask)*

In [26]:
# your turn


---

## ⚡ Extras — power tools, not required

**Several tag flags at once — `assign` + `apply` + `lambda`.** `.apply(f)` runs a function on every value of a column; `lambda t: "SmallParts" in t` is that function written inline (each value here is a list, and `in` you know from the primer); `.assign(...)` derives several columns in one call. Result: one row per item, safe to join:

In [27]:
flags = (
    item_tags
    .assign(
        has_small_parts=item_tags["tags"].apply(lambda t: "SmallParts" in t),
        has_oversized=item_tags["tags"].apply(lambda t: "Oversized" in t),
    )
    [["item_id", "has_small_parts", "has_oversized"]]
)
print("item_id unique in flags:", flags["item_id"].is_unique)

picks_flagged = picks.merge(flags, on="item_id", how="left")
assert len(picks_flagged) == len(picks)
print(f"rows before: {len(picks):,}   rows after: {len(picks_flagged):,}")

item_id unique in flags: True
rows before: 179,221   rows after: 179,221


**`validate=` — the tripwire built into `merge`.** Declare the relationship you *expect*: `"m:1"` means "many rows on my side may match at most one row on the reference side". If the reference key turns out non-unique, `merge` raises an error instead of handing you a longer table:

In [28]:
# validate="m:1" makes pandas check that the key is unique
# on the right side — and raise instead of returning a longer table
ok = picks.merge(
    flags,
    on="item_id",
    how="left",
    validate="m:1",
)
print("m:1 validated,", f"{len(ok):,}", "rows")

# the same line against the exploded table raises MergeError:
# picks.merge(tags_long, on="item_id", how="left", validate="m:1")

m:1 validated, 179,221 rows


**`right` join in the wild.** You'll rarely see it: `a.merge(b, how="right")` is just `b.merge(a, how="left")` with the columns in a different order — most people keep the table they care about on the left.

## Recap

- A join is VLOOKUP with two differences: it returns **every** match, not the first, and it can **delete** rows, not only add columns.
- Four join types: `inner` (matched keys only — **the default**), `left` (all my rows, honest gaps), `right`, `outer`. **An empty key never matches.**
- A join never errors — it hands you a *plausible wrong table*. **Count rows before and after, every time**; `assert` is the one-line tripwire.
- Failure #1, lost rows: `inner` drops unmatched keys and says nothing. `left` + `indicator=True` shows what didn't match instead of deleting it.
- Failure #2, multiplied rows: a non-unique key on the other side returns every combination. Check keys with `.is_unique`.
- Failure #3, the wrong key: the row count looks fine and the answer is nonsense. Row counting cannot catch it — say what the key means on each side before you join.
- A yes/no question about another table often needs no join at all: `.isin()` + a mask.
- A catalogue is not the traffic: week stats describe what moved.